# EV Charging Data Cleaning and Analysis

Step-by-step data cleaning and filtering of Electric Vehicle charging dataset from municipal lots and garages.

In [2]:
import pandas as pd
import numpy as np

# Step 1: Load the dataset
df = pd.read_csv('D:\\Research\\Research\\Project-EV\\Data\\Electric_Vehicle__EV__Charging_Data-_Municipal_Lots_and_Garages.csv')

print("=" * 80)
print("STEP 1: INITIAL DATA CHECK")
print("=" * 80)
print(f"\nTotal Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print(f"\nDataset Shape: {df.shape}")

# Store original values for later comparison
original_rows = len(df)
original_cols = df.shape[1]

print(f"\n{'Column Names:':}")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

print(f"\n{'Missing Values by Column:':}")
missing_values = df.isna().sum()
print(missing_values)

print(f"\nTotal Missing Values: {missing_values.sum()}")

STEP 1: INITIAL DATA CHECK

Total Rows: 211384
Total Columns: 15

Dataset Shape: (211384, 15)

Column Names:
  1. Date
  2. Station ID
  3. Location Name
  4. Country
  5. Charge Box ID
  6. Connector ID
  7. Driver ID
  8. ID Tag
  9. Connected Time
  10. Disconnected Time
  11. Charge Duration (min)
  12. Connected Duration (min)
  13. Energy Provided (kWh)
  14. Session Status
  15. Invalidity Reason

Missing Values by Column:
Date                             0
Station ID                       0
Location Name                   29
Country                      80459
Charge Box ID                    0
Connector ID                     0
Driver ID                    10217
ID Tag                           0
Connected Time                   0
Disconnected Time                0
Charge Duration (min)            0
Connected Duration (min)         0
Energy Provided (kWh)            0
Session Status                   0
Invalidity Reason           211384
dtype: int64

Total Missing Values: 30208

In [3]:
df['Location Name'].nunique()

19

In [4]:
df['Location Name'].unique()

array(['JGU - Jerome Gun Hill Road Municipal Parking Garage',
       'BRI - Bay Ridge Municipal Parking Garage',
       'DES - Delancey and Essex Municipal Parking Garage',
       'QBO - Queens Borough Hall Municipal Parking Garage',
       'JON - Jerome 190th Street Municipal Parking',
       'SGE - St. George Courthouse Garage', 'St. George Courthouse',
       'CSQ - Court Square Municipal Parking Garage',
       'QFA - Queens Family Court Municipal Garage',
       'Delancey and Essex Municipal Parking Garage',
       'Court Square Municipal Parking Garage', 'Queensboro Hall',
       'Jerome 190th Street Municipal Parking',
       'Bay Ridge Municipal Parking Garage',
       'Queens Family Court Municipal Garage',
       'Queens Borough Hall Municipal Parking Garage',
       'Jerome-Gun Hill Road Municipal Parking Garage', nan,
       'JON - Jerome 190th Street Municipal Parking Garage',
       'SGE - St. George Courthouse Municipal Parking Garage'],
      dtype=object)

In [ ]:

# Load original dataset
df_raw = pd.read_csv("Data/Electric_Vehicle__EV__Charging_Data-_Municipal_Lots_and_Garages.csv")

# Parse Date safely
date_parsed = pd.to_datetime(df_raw["Date"], format="mixed", errors="coerce")

print("ORIGINAL DATASET DATE RANGE")
print("-" * 40)
print(f"Total rows: {len(df_raw):,}")
print(f"Valid date rows: {date_parsed.notna().sum():,}")
print(f"Invalid date rows (NaT): {date_parsed.isna().sum():,}")
print(f"Earliest date: {date_parsed.min()}")
print(f"Latest date:   {date_parsed.max()}")

ORIGINAL DATASET DATE RANGE
----------------------------------------
Total rows: 211,384
Valid date rows: 211,384
Invalid date rows (NaT): 0
Earliest date: 2021-07-31 00:00:00
Latest date:   2025-12-16 00:00:00


In [7]:
print("\n" + "=" * 80)
print("STEP 2: KEEP REQUIRED COLUMNS ONLY")
print("=" * 80)

# Define required columns
required_columns = [
    'Date',
    'Station ID',
    'Location Name',
    'Connected Time',
    'Disconnected Time',
    'Charge Duration (min)',
    'Connected Duration (min)',
    'Energy Provided (kWh)'
]

# Check which columns exist
print(f"\nRequired columns: {required_columns}")
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    print(f"⚠️ Missing columns: {missing_cols}")

# Keep only required columns
df_filtered = df[required_columns].copy()

rows_after_column_filter = len(df_filtered)
columns_removed = original_cols - len(required_columns)

print(f"\nOriginal Dataset Shape: ({original_rows}, {original_cols})")
print(f"After Column Selection Shape: {df_filtered.shape}")
print(f"Columns Removed: {columns_removed}")
print(f"Columns Kept: {len(required_columns)}")


STEP 2: KEEP REQUIRED COLUMNS ONLY

Required columns: ['Date', 'Station ID', 'Location Name', 'Connected Time', 'Disconnected Time', 'Charge Duration (min)', 'Connected Duration (min)', 'Energy Provided (kWh)']

Original Dataset Shape: (211384, 15)
After Column Selection Shape: (211384, 8)
Columns Removed: 7
Columns Kept: 8


In [8]:
print("\n" + "=" * 80)
print("STEP 3: REMOVE MISSING VALUES")
print("=" * 80)

# Check missing values before removal
print(f"\nMissing values before dropna:")
print(df_filtered.isna().sum())

rows_before_dropna = len(df_filtered)

# Remove rows with any missing values
df_no_missing = df_filtered.dropna().copy()

rows_after_dropna = len(df_no_missing)
rows_removed_step3 = rows_before_dropna - rows_after_dropna

print(f"\nBefore dropna: {rows_before_dropna} rows")
print(f"After dropna: {rows_after_dropna} rows")
print(f"Rows Removed (Missing Values): {rows_removed_step3}")


STEP 3: REMOVE MISSING VALUES

Missing values before dropna:
Date                         0
Station ID                   0
Location Name               29
Connected Time               0
Disconnected Time            0
Charge Duration (min)        0
Connected Duration (min)     0
Energy Provided (kWh)        0
dtype: int64

Before dropna: 211384 rows
After dropna: 211355 rows
Rows Removed (Missing Values): 29


In [12]:
print("\n" + "=" * 80)
print("STEP 4: FILTER BY DATE RANGE AND SORT")
print("=" * 80)

# Convert Date column to datetime
df_no_missing['Date'] = pd.to_datetime(df_no_missing['Date'], format='%m/%d/%Y', errors='coerce')

# Display date range before filtering
print(f"\nDate range in dataset:")
print(f"  Earliest: {df_no_missing['Date'].min()}")
print(f"  Latest: {df_no_missing['Date'].max()}")

# Filter date range: 01/01/2021 to 12/15/2025
start_date = pd.to_datetime('01/01/2021', format='%m/%d/%Y')
end_date = pd.to_datetime('12/15/2025', format='%m/%d/%Y')

rows_before_date_filter = len(df_no_missing)

df_date_filtered = df_no_missing[(df_no_missing['Date'] >= start_date) & (df_no_missing['Date'] <= end_date)].copy()

# Sort by Date in descending order
df_final = df_date_filtered.sort_values('Date', ascending=False).reset_index(drop=True)

rows_after_date_filter = len(df_final)
rows_removed_step4 = rows_before_date_filter - rows_after_date_filter

print(f"\nDate Filter Applied: {start_date.date()} to {end_date.date()}")
print(f"Before date filter: {rows_before_date_filter} rows")
print(f"After date filter: {rows_after_date_filter} rows")
print(f"Rows Removed (Date Filtering): {rows_removed_step4}")
print(f"\nData sorted by Date in DESCENDING order")
print(f"  Start date requested: {start_date.date()}")
if len(df_final) > 0:
    print(f"  Earliest available record: {df_final['Date'].min()}")
    print(f"  Latest available record: {df_final['Date'].max()}")
else:
    print("  No records in the selected date range")


STEP 4: FILTER BY DATE RANGE AND SORT

Date range in dataset:
  Earliest: 2021-07-31 00:00:00
  Latest: 2025-12-16 00:00:00

Date Filter Applied: 2021-01-01 to 2025-12-15
Before date filter: 211355 rows
After date filter: 211324 rows
Rows Removed (Date Filtering): 31

Data sorted by Date in DESCENDING order
  Start date requested: 2021-01-01
  Earliest available record: 2021-07-31 00:00:00
  Latest available record: 2025-12-15 00:00:00


In [16]:
print("\n" + "=" * 80)
print("STEP 5: FINAL SUMMARY")
print("=" * 80)

# Calculate totals for summary
total_rows_removed = original_rows - len(df_final)

print(f"\n{'SUMMARY TABLE':}")
print("-" * 80)
print(f"{'Metric':<40} {'Rows':>15}")
print("-" * 80)
print(f"{'1. Original Dataset':<40} {original_rows:>15,}")
print(f"{'2. After Column Filtering':<40} {rows_after_column_filter:>15,}")
print(f"{'3. After Removing Missing Values':<40} {rows_after_dropna:>15,}")
print(f"{'4. After Date Filtering (01/01/2021-12/15/2025)':<40} {len(df_final):>15,}")
print("-" * 80)
print(f"{'TOTAL ROWS REMOVED':<40} {total_rows_removed:>15,}")
print(f"{'FINAL DATASET SIZE':<40} {len(df_final):>15,}")
print("-" * 80)

print(f"\nFinal Dataset Shape: {df_final.shape}")
print(f"  Rows: {df_final.shape[0]:,}")
print(f"  Columns: {df_final.shape[1]}")

print(f"\n{'BREAKDOWN OF ROWS REMOVED BY STEP':}")
print(f"  Step 1 (Load): 0 rows removed")
print(f"  Step 2 (Column Selection): 0 rows removed")
print(f"  Step 3 (Missing Values): {rows_removed_step3:,} rows removed")
print(f"  Step 4 (Date Filter): {rows_removed_step4:,} rows removed")
print(f"  {'─' * 50}")
print(f"  Total Removed: {total_rows_removed:,} rows ({(total_rows_removed/original_rows*100):.2f}%)")




STEP 5: FINAL SUMMARY

SUMMARY TABLE
--------------------------------------------------------------------------------
Metric                                              Rows
--------------------------------------------------------------------------------
1. Original Dataset                              211,384
2. After Column Filtering                        211,384
3. After Removing Missing Values                 211,355
4. After Date Filtering (01/01/2021-12/15/2025)         211,324
--------------------------------------------------------------------------------
TOTAL ROWS REMOVED                                    60
FINAL DATASET SIZE                               211,324
--------------------------------------------------------------------------------

Final Dataset Shape: (211324, 8)
  Rows: 211,324
  Columns: 8

BREAKDOWN OF ROWS REMOVED BY STEP
  Step 1 (Load): 0 rows removed
  Step 2 (Column Selection): 0 rows removed
  Step 3 (Missing Values): 29 rows removed
  Step 4 (Date 

In [15]:
print("\n" + "=" * 80)
print("STEP 6: SAVE CLEANED DATASET")
print("=" * 80)

output_path = 'Data/EV_Charging_Cleaned.csv'
df_final.to_csv(output_path, index=False)

print(f"Saved file: {output_path}")
print(f"Rows saved: {len(df_final):,}")
print(f"Columns saved: {df_final.shape[1]}")


STEP 6: SAVE CLEANED DATASET
Saved file: Data/EV_Charging_Cleaned.csv
Rows saved: 211,324
Columns saved: 8


In [19]:
EV = pd.read_csv('Data/EV_Charging_Cleaned.csv')

In [24]:
EV.head(232)

,Date,Station ID,Location Name,Connected Time,Disconnected Time,Charge Duration (min),Connected Duration (min),Energy Provided (kWh)
0,12/15/2025,EVB3052,CSQ - Court Square Municipal Parking Garage,8:13:47 AM,5:29:06 PM,502.116667,555.316667,50.737
1,12/15/2025,101337,QBO - Queens Borough Hall Municipal Parking Ga...,2:01:30 PM,2:48:54 PM,47.403200,47.403200,39.629
2,12/15/2025,101337,QBO - Queens Borough Hall Municipal Parking Ga...,5:31:00 PM,6:04:43 PM,33.715417,33.715417,14.229
3,12/15/2025,EV0444,CSQ - Court Square Municipal Parking Garage,6:49:23 PM,7:43:02 PM,53.650000,53.650000,30.876
4,12/15/2025,101337,QBO - Queens Borough Hall Municipal Parking Ga...,4:49:16 PM,5:28:23 PM,39.123100,39.123100,23.088
...,...,...,...,...,...,...,...,...
227,12/15/2025,EVB3052,CSQ - Court Square Municipal Parking Garage,7:35:14 PM,1:16:53 AM,217.483333,341.650000,22.458
228,12/15/2025,EVB3047,CSQ - Court Square Municipal Parking Garage,6:11:54 PM,10:20:20 PM,248.433333,248.433333,16.923
229,12/15/2025,EVB3045,CSQ - Court Square Municipal Parking Garage,12:25:31 PM,3:31:24 PM,185.883333,185.883333,19.970
230,12/14/2025,101337,QBO - Queens Borough Hall Municipal Parking Ga...,7:51:58 AM,8:12:19 AM,20.351733,20.351733,22.158


In [25]:
df.tail(100)

,Date,Station ID,Location Name,Country,Charge Box ID,Connector ID,Driver ID,ID Tag,Connected Time,Disconnected Time,Charge Duration (min),Connected Duration (min),Energy Provided (kWh),Session Status,Invalidity Reason
211284,12/15/2025,EV0437,CSQ - Court Square Municipal Parking Garage,USA,EV0437,1,242761fb-b234-44ea-b256-2af3d21242c1,LZFHKQK5LB1Z0ASXBYHJ,5:56:10 AM,7:43:08 AM,106.966667,106.966667,21.596,PAID,NaN
211285,12/15/2025,EV0444,CSQ - Court Square Municipal Parking Garage,USA,EV0444,1,9e3c6526-25e5-440f-aee1-091a93c243da,29KHBZ75IX1EZ3BKHWV9,7:57:37 PM,8:53:19 PM,55.700000,55.700000,23.971,PAID,NaN
211286,12/15/2025,EV0444,CSQ - Court Square Municipal Parking Garage,USA,EV0444,1,e52e84f1-1f29-4dfa-8294-7230b19a58cf,9OYZ6KR6XKQHQU8FBFDI,4:28:51 PM,4:57:26 PM,28.583333,28.583333,19.658,PAID,NaN
211287,12/15/2025,EV0444,CSQ - Court Square Municipal Parking Garage,USA,EV0444,1,e52e84f1-1f29-4dfa-8294-7230b19a58cf,9OYZ6KR6XKQHQU8FBFDI,4:58:39 PM,5:26:48 PM,28.150000,28.150000,21.270,PAID,NaN
211288,12/15/2025,EV0444,CSQ - Court Square Municipal Parking Garage,USA,EV0444,1,bf8da4dc-fb92-4b60-ae9b-7a316ccb321b,c6df2d0d-9371-4188-9,6:57:00 AM,9:45:40 AM,168.666667,168.666667,28.833,ROAMING,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211379,12/16/2025,EV0445,DES - Delancey and Essex Municipal Parking Garage,USA,EV0445,1,172b8bae-1fa4-4721-863e-5bc697188e42,YJ9T72RJ38G7AFBSEANE,1:00:28 AM,2:17:06 AM,76.633333,76.633333,65.035,PAID,NaN
211380,12/16/2025,EV0449,DES - Delancey and Essex Municipal Parking Garage,USA,EV0449,2,23bb393c-2b0b-4ff7-9205-76d09dba0aa1,JQKOB1KQXM21J96NU9G6,12:53:13 AM,1:04:48 AM,11.583333,11.583333,4.294,PAID,NaN
211381,12/16/2025,EV0449,DES - Delancey and Essex Municipal Parking Garage,USA,EV0449,2,23bb393c-2b0b-4ff7-9205-76d09dba0aa1,JQKOB1KQXM21J96NU9G6,1:07:20 AM,2:05:03 AM,57.716667,57.716667,19.894,PAID,NaN
211382,12/16/2025,EV0449,DES - Delancey and Essex Municipal Parking Garage,USA,EV0449,2,23bb393c-2b0b-4ff7-9205-76d09dba0aa1,JQKOB1KQXM21J96NU9G6,2:05:53 AM,3:18:12 AM,72.316667,72.316667,19.459,PAID,NaN


## Asos Dataset Preprocessing

In [17]:
import pandas as pd

df = pd.read_csv('Data/asos.csv')
df.head()

C:\Users\User\AppData\Local\Temp\ipykernel_3920\2543381744.py:3: DtypeWarning: Columns (24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Data/asos.csv')


,station,valid,tmpf,dwpf,dwpc,relh,feel,drct,sknt,sped,...,wxcodes,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_gust_mph,peak_wind_drct,peak_wind_time,snowdepth,metar
0,NYC,1/1/2020 0:51,44.0,36.0,2.22,72.99,39.53,200.0,7.0,8.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNYC 010051Z AUTO 20007KT 10SM BKN049 OVC060 0...
1,LGA,1/1/2020 0:51,45.0,35.0,1.67,68.05,39.68,210.0,9.0,10.35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KLGA 010051Z 21009KT 10SM BKN050 OVC060 07/02 ...
2,JRB,1/1/2020 0:56,44.0,38.0,3.33,78.66,39.53,260.0,7.0,8.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KJRB 010056Z AUTO 26007KT OVC050 07/03 A2964 R...
3,LGA,1/1/2020 1:51,44.0,34.0,1.11,67.41,36.76,280.0,14.0,16.10,...,#NAME?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KLGA 010151Z 28014G24KT 10SM -RA FEW027 SCT033...
4,NYC,1/1/2020 1:51,43.0,35.0,1.67,73.45,36.00,280.0,12.0,13.80,...,#NAME?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNYC 010151Z AUTO 28012G20KT 260V320 9SM -RA S...


In [18]:
df.isnull().sum()

station                    0
valid                      0
tmpf                    8105
dwpf                    9315
dwpc                    9315
relh                    9315
feel                   10981
drct                   39069
sknt                    6230
sped                    6230
alti                     259
mslp                   41373
p01m                       6
p01i                       6
vsby                    1548
gust                  158240
gust_mph              158240
skyc1                    705
skyc2                 122154
skyc3                 159192
skyl1                  57014
skyl2                 122155
skyl3                 159192
wxcodes               150613
ice_accretion_1hr     189738
ice_accretion_3hr     189860
ice_accretion_6hr     189851
peak_wind_gust        176415
peak_wind_gust_mph    176415
peak_wind_drct        176415
peak_wind_time        176415
snowdepth             189591
metar                      0
dtype: int64

In [2]:
df.shape

(189874, 33)

In [3]:
# Check columns and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189874 entries, 0 to 189873
Data columns (total 33 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   station             189874 non-null  object 
 1   valid               189874 non-null  object 
 2   tmpf                181769 non-null  float64
 3   dwpf                180559 non-null  float64
 4   dwpc                180559 non-null  float64
 5   relh                180559 non-null  float64
 6   feel                178893 non-null  float64
 7   drct                150805 non-null  float64
 8   sknt                183644 non-null  float64
 9   sped                183644 non-null  float64
 10  alti                189615 non-null  float64
 11  mslp                148501 non-null  float64
 12  p01m                189868 non-null  object 
 13  p01i                189868 non-null  object 
 14  vsby                188326 non-null  float64
 15  gust                31634 non-null

In [4]:
print("=" * 80)
print("STEP 1: SELECT REQUIRED COLUMNS ONLY")
print("=" * 80)

# Define required columns
required_cols = ['station', 'valid', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']

# Select only required columns
df_filtered = df[required_cols].copy()

print(f"\nOriginal columns: {list(df.columns)}")
print(f"Selected columns: {list(df_filtered.columns)}")
print(f"Original shape: {df.shape}")
print(f"Filtered shape: {df_filtered.shape}")

# Rename 'valid' to 'Date' for clarity
df_filtered.rename(columns={'valid': 'Date'}, inplace=True)

print("\n" + "=" * 80)
print("STEP 2: CONVERT HOURLY DATA TO DAILY AGGREGATES")
print("=" * 80)

# Parse Date column
df_filtered['Date'] = pd.to_datetime(df_filtered['Date'], errors='coerce')
df_filtered['date'] = df_filtered['Date'].dt.date

# Convert numeric columns
df_filtered['tmpf'] = pd.to_numeric(df_filtered['tmpf'], errors='coerce')
df_filtered['relh'] = pd.to_numeric(df_filtered['relh'], errors='coerce')
df_filtered['feel'] = pd.to_numeric(df_filtered['feel'], errors='coerce')
df_filtered['sped'] = pd.to_numeric(df_filtered['sped'], errors='coerce')
df_filtered['p01m'] = pd.to_numeric(df_filtered['p01m'], errors='coerce')
df_filtered['snowdepth'] = pd.to_numeric(df_filtered['snowdepth'], errors='coerce')

# Group by station and date, then aggregate
df_daily = df_filtered.groupby(['station', 'date']).agg({
    'tmpf': 'mean',        # Average temperature
    'relh': 'mean',        # Average relative humidity
    'feel': 'mean',        # Average feels like temperature
    'sped': 'mean',        # Average wind speed
    'p01m': 'sum',         # Total precipitation
    'snowdepth': 'max'     # Maximum snow depth
}).reset_index()

# Convert snowdepth: if max > 0 then 1, else 0 (presence/absence)
df_daily['snowdepth'] = (df_daily['snowdepth'] > 0).astype(int)

print(f"\nAfter hourly to daily conversion:")
print(f"  Shape: {df_daily.shape}")
print(f"  Date range: {df_daily['date'].min()} to {df_daily['date'].max()}")
print(f"  Unique dates: {df_daily['date'].nunique()}")
print(f"  Unique stations: {df_daily['station'].nunique()}")

STEP 1: SELECT REQUIRED COLUMNS ONLY

Original columns: ['station', 'valid', 'tmpf', 'dwpf', 'dwpc', 'relh', 'feel', 'drct', 'sknt', 'sped', 'alti', 'mslp', 'p01m', 'p01i', 'vsby', 'gust', 'gust_mph', 'skyc1', 'skyc2', 'skyc3', 'skyl1', 'skyl2', 'skyl3', 'wxcodes', 'ice_accretion_1hr', 'ice_accretion_3hr', 'ice_accretion_6hr', 'peak_wind_gust', 'peak_wind_gust_mph', 'peak_wind_drct', 'peak_wind_time', 'snowdepth', 'metar']
Selected columns: ['station', 'valid', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']
Original shape: (189874, 33)
Filtered shape: (189874, 8)

STEP 2: CONVERT HOURLY DATA TO DAILY AGGREGATES

After hourly to daily conversion:
  Shape: (6551, 8)
  Date range: 2020-01-01 to 2026-01-30
  Unique dates: 2222
  Unique stations: 3


In [5]:
print("\n" + "=" * 80)
print("STEP 3: FINALIZE AND SAVE")
print("=" * 80)

# Round float columns to 2 decimal places
float_columns = df_daily.select_dtypes(include=['float64']).columns
df_daily[float_columns] = df_daily[float_columns].round(2)

# Reorder columns: date first, then station, then rest
df_daily = df_daily[['date', 'station', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']]

# Sort by date first, then by station
df_daily = df_daily.sort_values(['date', 'station']).reset_index(drop=True)

# Save to CSV file
df_daily.to_csv('Data/asos_daily_cleaned.csv', index=False)

print(f"\n✓ File saved: Data/asos_daily_cleaned.csv")
print(f"\nFinal Summary:")
print(f"  Total rows: {len(df_daily):,}")
print(f"  Unique dates: {df_daily['date'].nunique()}")
print(f"  Unique stations: {df_daily['station'].nunique()}")
print(f"  Columns: {list(df_daily.columns)}")
print(f"\nFirst 5 rows:")
df_daily.head()


STEP 3: FINALIZE AND SAVE

✓ File saved: Data/asos_daily_cleaned.csv

Final Summary:
  Total rows: 6,551
  Unique dates: 2222
  Unique stations: 3
  Columns: ['date', 'station', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']

First 5 rows:


,date,station,tmpf,relh,feel,sped,p01m,snowdepth
0,2020-01-01,JRB,39.62,60.72,32.79,11.16,0.00,0
1,2020-01-01,LGA,39.58,51.91,30.95,16.15,0.25,0
2,2020-01-01,NYC,39.02,55.07,32.95,9.06,0.51,0
3,2020-01-02,JRB,38.96,59.52,33.75,7.67,1.01,0
4,2020-01-02,LGA,40.12,48.92,34.17,9.49,0.00,0


In [4]:
print("=" * 80)
print("ASOS DATA CLEANING AND FILTERING")
print("=" * 80)

# Load ASOS dataset
asos = pd.read_csv('Data/asos_daily_cleaned.csv')

print(f"\n1. ORIGINAL ASOS DATA")
print(f"   Shape: {asos.shape}")
print(f"   Columns: {list(asos.columns)}")
print(f"   First few rows:")
print(asos.head(3))


ASOS DATA CLEANING AND FILTERING

1. ORIGINAL ASOS DATA
   Shape: (6551, 8)
   Columns: ['Date', 'station', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']
   First few rows:
       Date station   tmpf   relh   feel   sped  p01m  snowdepth
0  1/1/2020     JRB  39.62  60.72  32.79  11.16  0.00          0
1  1/1/2020     LGA  39.58  51.91  30.95  16.15  0.25          0
2  1/1/2020     NYC  39.02  55.07  32.95   9.06  0.51          0


In [5]:
# Define required columns
required_asos_cols = ['station', 'Date', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']

# Check which columns exist
existing_cols = [col for col in required_asos_cols if col in asos.columns]
missing_cols = [col for col in required_asos_cols if col not in asos.columns]

print(f"\n2. COLUMN SELECTION")
print(f"   Required columns: {required_asos_cols}")
print(f"   Existing columns: {existing_cols}")
if missing_cols:
    print(f"   ⚠️ Missing columns: {missing_cols}")

# Keep only existing required columns
asos_filtered = asos[existing_cols].copy()
print(f"   After column selection: {asos_filtered.shape}")



2. COLUMN SELECTION
   Required columns: ['station', 'Date', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']
   Existing columns: ['station', 'Date', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']
   After column selection: (6551, 8)


In [6]:
# Parse Date column
asos_filtered['Date'] = pd.to_datetime(asos_filtered['Date'], errors='coerce')

# Remove rows with NaT dates
asos_valid = asos_filtered[asos_filtered['Date'].notna()].copy()

print(f"\n3. DATE PARSING")
print(f"   Rows with valid dates: {len(asos_valid):,} (removed {len(asos_filtered) - len(asos_valid):,})")
print(f"   Date range in data:")
print(f"      Earliest: {asos_valid['Date'].min()}")
print(f"      Latest:   {asos_valid['Date'].max()}")

# Filter by EV date range: 12/15/2025 to 2021-07-31
ev_start = pd.to_datetime('12/15/2025')
ev_end = pd.to_datetime('2021-07-31')

asos_date_filtered = asos_valid[(asos_valid['Date'] <= ev_start) & (asos_valid['Date'] >= ev_end)].copy()

print(f"\n4. DATE RANGE FILTERING")
print(f"   Filter range: {ev_start.date()} to {ev_end.date()}")
print(f"   Rows matching date range: {len(asos_date_filtered):,}")
print(f"   Rows removed: {len(asos_valid) - len(asos_date_filtered):,}")



3. DATE PARSING
   Rows with valid dates: 6,551 (removed 0)
   Date range in data:
      Earliest: 2020-01-01 00:00:00
      Latest:   2026-01-30 00:00:00

4. DATE RANGE FILTERING
   Filter range: 2025-12-15 to 2021-07-31
   Rows matching date range: 4,683
   Rows removed: 1,868


In [7]:
# Reorder columns: Date first, then others
reordered_cols = ['Date'] + [col for col in existing_cols if col != 'Date']
asos_final = asos_date_filtered[reordered_cols].copy()

# Sort by Date in descending order (matching EV data)
asos_final = asos_final.sort_values('Date', ascending=False).reset_index(drop=True)

print(f"\n5. FINAL ASOS DATASET")
print(f"   Shape: {asos_final.shape}")
print(f"   Columns (Date first): {list(asos_final.columns)}")
print(f"   Date range (sorted descending):")
print(f"      Latest:   {asos_final['Date'].max()}")
print(f"      Earliest: {asos_final['Date'].min()}")
print(f"\n   First 5 rows:")
print(asos_final.head())



5. FINAL ASOS DATASET
   Shape: (4683, 8)
   Columns (Date first): ['Date', 'station', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']
   Date range (sorted descending):
      Latest:   2025-12-15 00:00:00
      Earliest: 2021-07-31 00:00:00

   First 5 rows:
        Date station   tmpf   relh   feel   sped   p01m  snowdepth
0 2025-12-15     NYC  23.04  54.41  13.05   9.78   0.00          0
1 2025-12-15     LGA  24.46  49.68  11.19  17.01   0.00          1
2 2025-12-15     JRB  26.12  66.88  20.25   5.61   0.00          0
3 2025-12-14     NYC  30.38  78.11  24.95   5.26  13.72          0
4 2025-12-14     LGA  31.73  72.02  24.85   9.36  18.54          1


In [8]:
# Save cleaned ASOS dataset
asos_output_path = 'Data/ASOS_Cleaned.csv'
asos_final.to_csv(asos_output_path, index=False)

print(f"\n6. SAVE CLEANED ASOS DATASET")
print(f"   Saved file: {asos_output_path}")
print(f"   Rows saved: {len(asos_final):,}")
print(f"   Columns saved: {asos_final.shape[1]}")
print(f"   Columns: {list(asos_final.columns)}")



6. SAVE CLEANED ASOS DATASET
   Saved file: Data/ASOS_Cleaned.csv
   Rows saved: 4,683
   Columns saved: 8
   Columns: ['Date', 'station', 'tmpf', 'relh', 'feel', 'sped', 'p01m', 'snowdepth']


In [9]:
asos = pd.read_csv('Data/ASOS_Cleaned.csv')

In [10]:
asos.head(4)

,Date,station,tmpf,relh,feel,sped,p01m,snowdepth
0,2025-12-15,NYC,23.04,54.41,13.05,9.78,0.00,0
1,2025-12-15,LGA,24.46,49.68,11.19,17.01,0.00,1
2,2025-12-15,JRB,26.12,66.88,20.25,5.61,0.00,0
3,2025-12-14,NYC,30.38,78.11,24.95,5.26,13.72,0


## Merge EV_Charging_Cleane.csv and ASOS_Cleaned.csv

In [2]:
import pandas as pd

df_ev = pd.read_csv("D:/Research/Research/Project-EV/Data/EV_Charging_Cleaned.csv")
df_weather = pd.read_csv("D:/Research/Research/Project-EV/Data/ASOS_Cleaned.csv")

In [3]:
df_ev["Date"] = pd.to_datetime(df_ev["Date"]).dt.date
df_weather["Date"] = pd.to_datetime(df_weather["Date"]).dt.date

In [4]:
location_weather_map = {
    # Queens → LGA
    "CSQ - Court Square Municipal Parking Garage": "LGA",
    "QBO - Queens Borough Hall Municipal Parking Garage": "LGA",
    "QFA - Queens Family Court Municipal Garage": "LGA",
    "Queens Borough Hall Municipal Parking Garage": "LGA",
    "Queens Family Court Municipal Garage": "LGA",
    "Queensboro Hall": "LGA",
    "Court Square Municipal Parking Garage": "LGA",

    # Bronx → JRB
    "JON - Jerome 190th Street Municipal Parking Garage": "JRB",
    "JON - Jerome 190th Street Municipal Parking": "JRB",
    "Jerome 190th Street Municipal Parking": "JRB",
    "JGU - Jerome Gun Hill Road Municipal Parking Garage": "JRB",
    "Jerome-Gun Hill Road Municipal Parking Garage": "JRB",

    # Manhattan / Brooklyn / Staten Island → NYC
    "DES - Delancey and Essex Municipal Parking Garage": "NYC",
    "Delancey and Essex Municipal Parking Garage": "NYC",
    "BRI - Bay Ridge Municipal Parking Garage": "NYC",
    "Bay Ridge Municipal Parking Garage": "NYC",
    "SGE - St. George Courthouse Municipal Parking Garage": "NYC",
    "SGE - St. George Courthouse Garage": "NYC",
    "St. George Courthouse": "NYC"
}

df_ev["weather_station"] = df_ev["Location Name"].map(location_weather_map)

In [5]:
df_ev["weather_station"].value_counts()

weather_station
LGA    118421
NYC     73684
JRB     19219
Name: count, dtype: int64

In [6]:
df_final = df_ev.merge(
    df_weather,
    left_on=["Date", "weather_station"],
    right_on=["Date", "station"],
    how="left"
)

In [7]:
df_final.drop(columns=["station"], inplace=True)

In [8]:
df_final.to_csv("Data/EV.csv", index=False)

In [9]:
ev = pd.read_csv("Data/EV.csv")

In [10]:
ev.head()

,Date,Station ID,Location Name,Connected Time,Disconnected Time,Charge Duration (min),Connected Duration (min),Energy Provided (kWh),weather_station,tmpf,relh,feel,sped,p01m,snowdepth
0,2025-12-15,EVB3052,CSQ - Court Square Municipal Parking Garage,8:13:47 AM,5:29:06 PM,502.116667,555.316667,50.737,LGA,24.46,49.68,11.19,17.01,0.0,1.0
1,2025-12-15,101337,QBO - Queens Borough Hall Municipal Parking Ga...,2:01:30 PM,2:48:54 PM,47.403200,47.403200,39.629,LGA,24.46,49.68,11.19,17.01,0.0,1.0
2,2025-12-15,101337,QBO - Queens Borough Hall Municipal Parking Ga...,5:31:00 PM,6:04:43 PM,33.715417,33.715417,14.229,LGA,24.46,49.68,11.19,17.01,0.0,1.0
3,2025-12-15,EV0444,CSQ - Court Square Municipal Parking Garage,6:49:23 PM,7:43:02 PM,53.650000,53.650000,30.876,LGA,24.46,49.68,11.19,17.01,0.0,1.0
4,2025-12-15,101337,QBO - Queens Borough Hall Municipal Parking Ga...,4:49:16 PM,5:28:23 PM,39.123100,39.123100,23.088,LGA,24.46,49.68,11.19,17.01,0.0,1.0


In [11]:
ev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211324 entries, 0 to 211323
Data columns (total 15 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Date                      211324 non-null  object 
 1   Station ID                211324 non-null  object 
 2   Location Name             211324 non-null  object 
 3   Connected Time            211324 non-null  object 
 4   Disconnected Time         211324 non-null  object 
 5   Charge Duration (min)     211324 non-null  float64
 6   Connected Duration (min)  211324 non-null  float64
 7   Energy Provided (kWh)     211324 non-null  float64
 8   weather_station           211324 non-null  object 
 9   tmpf                      209285 non-null  float64
 10  relh                      209187 non-null  float64
 11  feel                      209187 non-null  float64
 12  sped                      210629 non-null  float64
 13  p01m                      210910 non-null  f

In [12]:
ev.shape

(211324, 15)

In [15]:
ev.isnull().sum()

Date                           0
Station ID                     0
Location Name                  0
Connected Time                 0
Disconnected Time              0
Charge Duration (min)          0
Connected Duration (min)       0
Energy Provided (kWh)          0
weather_station                0
tmpf                        2039
relh                        2137
feel                        2137
sped                         695
p01m                         414
snowdepth                    414
dtype: int64

In [19]:
print(ev.shape)
print(ev.info())
print(ev.isnull().sum())

(211324, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211324 entries, 0 to 211323
Data columns (total 15 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Date                      211324 non-null  object 
 1   Station ID                211324 non-null  object 
 2   Location Name             211324 non-null  object 
 3   Connected Time            211324 non-null  object 
 4   Disconnected Time         211324 non-null  object 
 5   Charge Duration (min)     211324 non-null  float64
 6   Connected Duration (min)  211324 non-null  float64
 7   Energy Provided (kWh)     211324 non-null  float64
 8   weather_station           211324 non-null  object 
 9   tmpf                      209285 non-null  float64
 10  relh                      209187 non-null  float64
 11  feel                      209187 non-null  float64
 12  sped                      210629 non-null  float64
 13  p01m                      21091

In [20]:
ev["Date"] = pd.to_datetime(ev["Date"])
ev = ev.sort_values(["weather_station", "Date"])

In [22]:
weather_cols = ["tmpf","relh","feel","sped","p01m","snowdepth"]

ev = ev.sort_values(["weather_station","Date"])

ev[weather_cols] = (
    ev.groupby("weather_station")[weather_cols]
    .transform(lambda x: x.interpolate())
)

ev[weather_cols] = ev[weather_cols].bfill()
ev[weather_cols] = ev[weather_cols].ffill()

In [24]:

print(ev.isnull().sum())

Date                        0
Station ID                  0
Location Name               0
Connected Time              0
Disconnected Time           0
Charge Duration (min)       0
Connected Duration (min)    0
Energy Provided (kWh)       0
weather_station             0
tmpf                        0
relh                        0
feel                        0
sped                        0
p01m                        0
snowdepth                   0
dtype: int64
